# Figure 4 | lncRNA QTLs contribute GWAS heritability and colocalize with trait-associated loci

**a.** Heritability enrichment of cis-eQTL/cis-sQTL annotations for protein-coding genes, protein-coding genes down-sampled to match lncRNA SNP density, all lncRNAs, existing lncRNAs, and novel lncRNAs from S-LDSC (all tissues except testis). Points, inverse-variance-weighted (IVW) meta-analytic mean enrichment across 87 genome-wide association study (GWAS) traits; error bars, 95% confidence intervals (CIs). Dotted line, enrichment = 1. Significance, two-sided test of enrichment ≠ 1 (*, p < 0.05; **, p < 0.01; ***, p < 0.001). Novel lncRNA cis-sQTL annotations are sparse and are excluded (X). lncRNA cis-eQTL but not cis-sQTL annotations are significantly enriched.

**b.** Conditional effect size τ* for the biotype-stratified cis-eQTL/cis-sQTL annotations in a. Open symbols, QTL (marginal); light-filled symbols, QTL | protein-coding e/sQTL (lncRNAs only); filled symbols, QTL | LDSC baseline. Marker shapes and colors as in a. Error bars, 95% CIs. Significance, one-sided test of τ* > 0 (stars as in a). lncRNA cis-eQTL effects remain positive after conditioning on protein-coding QTLs but are attenuated to nonsignificance after the full LDSC baseline.

**c.** UpSet of cross-tissue merged GWAS-QTL CoS loci by colocalizing biotype: protein-coding, existing lncRNA, novel lncRNA, and pseudogene. Light grey bars, loci with GWAS p < 10-5; dark grey bars, genome-wide significant subset (GWAS p < 5.7*10-9). Colored boxes mark lncRNA-only intersections: novel-only (red); lncRNA-only which is existing-only and/or novel and existing (pink).

**d.** Unique genes with ≥1 GWAS-QTL colocalized set (CoS; GWAS p < 10-5) for novel lncRNA and existing lncRNA. Outlined bars, genes in any CoS; solid bars, genes in a CoS whose colocalizing genes are restricted to novel lncRNAs for novel lncRNAs (novel-lncRNA specific) or novel and/or existing lncRNAs for existing lncRNA (lncRNA-specific).

**e.** Logistic odds ratios (ORs) for membership of annotated lncRNAs in a GWAS-QTL CoS (GWAS p < 10-5), comparing lncRNA-specific vs shared-with-non-lncRNA CoS. Models include ascertainment covariates: numbers of tissues tested for eQTL and sQTL, and expression as log10(TPM + 1) where TPM is the maximum median tissue TPM. Categories are ordered by mean of lncRNA-specific and shared OR. Categories: tissue-concordant expression with non-testis peak organ (TC, non-testis); developmental expression (dGTEx expression at FDR 5% vs intergenic control); tissue-specific expression with non-testis peak organ (TS, non-testis); miRNA host; primate splice-junction conservation (unique constrained bases); tissue-specific expression with testis peak organ (TS, testis); translated non-canonical ORF (ncORF; ribo-seq); primate exon conservation (unique constrained bases); enhancer (transcription start overlap E2G); primate expression (NHP-dGTEx at FDR 5% vs intergenic control); and tissue-concordant expression with testis peak organ (TC, testis). Points, ORs; error bars, 95% CIs; asterisks, two-sided Wald p (* p < 0.05; ** p < 0.01; *** p < 0.001). Grey ×, model not estimable (no TC-testis genes in the lncRNA-only CoS).


## Setup


In [ ]:
import gzip
import json
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle
from scipy import stats
from tqdm.auto import tqdm

plt.rc("axes", labelsize=6)
plt.rc("xtick", labelsize=5)
plt.rc("ytick", labelsize=6)
plt.rc("legend", fontsize=4)
plt.rc("legend", title_fontsize=5)
plt.rc("font", size=6)
plt.rc("figure", titlesize=7)
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["figure.figsize"] = (7.2, 5.0)
plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300

expected_n_bulk_tissues = 54  # histology subtypes excluded; includes 2 cell lines

# Max-TPM covariate uses the 54 bulk sites (not the 52 τ set): drop histology subtypes only
histology_subtypes = {
    "Colon - Transverse - Muscularis",
    "Stomach - Mixed Cell",
    "Pancreas - Mixed Cell",
    "Small Intestine - Terminal Ileum - Mixed Cell",
    "Stomach - Mucosa",
    "Stomach - Muscularis",
    "Pancreas - Acini",
    "Small Intestine - Terminal Ileum - Lymphoid Aggregate",
    "Colon - Transverse - Mixed Cell",
    "Colon - Transverse - Mucosa",
    "Liver - Mixed Cell",
    "Pancreas - Islets",
    "Liver - Hepatocyte",
    "Liver - Portal Tract",
}
biotype_model_suffix = "_noTestis"
qtl_marker = {"eqtl": "o", "sqtl": "s"}
star_fs = 4
ds_rep = "protein_coding_downsampled"
gwas_p_cut_1e5 = 1e-5
gwas_neglog10p_1e5 = -np.log10(gwas_p_cut_1e5)

ensg_stem = re.compile(r"(ENSG\d+)")

gene_type_palette = {
    "novel_lncRNA": sns.color_palette("Paired", 12)[5],
    "existing_lncRNA": sns.color_palette("Paired", 12)[4],
    "lncRNA_all": sns.color_palette("Reds", 7)[5],
    "protein_coding": sns.color_palette("Paired", 12)[1],
    "protein_coding_downsampled": sns.color_palette("Blues", 7)[4],
    "pseudogene": sns.color_palette("Paired", 12)[3],
}
# Shared c/d colors: novel-only / existing lncRNA-only use biotype colors; other CoS = white
cos_slice_colors = {
    "novel_only": gene_type_palette["novel_lncRNA"],
    "existing_only": gene_type_palette["existing_lncRNA"],
    "novel_existing": gene_type_palette["existing_lncRNA"],  # same as existing (lncRNA-only)
    "shared": "white",
}
biotype_groups = [
    ("protein_coding", "protein\ncoding"),
    ("protein_coding_downsampled", "down\nsampled\nprotein\ncoding"),
    ("lncRNA_all", "all\nlncRNA"),
    ("existing_lncRNA", "existing\nlncRNA"),
    ("novel_lncRNA", "novel\nlncRNA"),
]

qtl_label_for_figure = {
    "no_qtl": "no_qtl",
    "non-lncRNA": "non-lncRNA",
    "Shared": "Shared",
    "lncRNA-specific (weak)": "Shared",
    "lncRNA-specific (robust)": "lncRNA-specific",
}

# Zenodo ColocBoost helpers
def tissue_from_cos_id(cid):
    s = str(cid)
    i = s.rfind("_")
    return s[:i] if i > 0 else s


def lead_from_lists(variants, vcp):
    vs = list(variants)
    vcps = list(vcp)
    ps = pd.to_numeric(pd.Series(vcps), errors="coerce")
    if len(ps) == len(vs) and ps.notna().any():
        return vs[int(ps.values.argmax())]
    return vs[0]


def cos_to_long(row):
    vs = list(row["variants"])
    ps = list(row["vcp"])
    return pd.DataFrame({
        "variant_id": vs,
        "vcp": ps[: len(vs)] + [float("nan")] * (len(vs) - len(ps)),
    })


def _coloc_list_len(val):
    if val is None:
        return 0
    if isinstance(val, float) and np.isnan(val):
        return 0
    if isinstance(val, np.ndarray):
        return len(val)
    if isinstance(val, (list, tuple, set)):
        return len(val)
    return 0


def _typeset(vals):
    if vals is None:
        return set()
    if isinstance(vals, float) and np.isnan(vals):
        return set()
    if isinstance(vals, np.ndarray):
        items = vals.tolist()
    elif isinstance(vals, (list, tuple, set)):
        items = list(vals)
    else:
        items = [vals]
    out = set()
    for v in items:
        if v is None:
            continue
        s = str(v).strip()
        if s and s not in ("nan", "None", "<NA>"):
            out.add(s)
    return out


def _member_qtl_class(types):
    t = _typeset(types)
    lnc = t & {"novel_lncRNA", "existing_lncRNA"}
    non = t - {"novel_lncRNA", "existing_lncRNA"}
    if not lnc:
        return "non-lncRNA"
    if non:
        return "Shared"
    return "lncRNA-specific (robust)"


def _locus_qtl_label(member_classes):
    labs = set(member_classes)
    if labs == {"lncRNA-specific (robust)"}:
        return "lncRNA-specific (robust)"
    if "Shared" in labs or ("non-lncRNA" in labs and "lncRNA-specific (robust)" in labs):
        return "Shared"
    return "non-lncRNA"


def load_main_colocboost(colocboost_dir, *, cols=None):
    frames = []
    for path in sorted(colocboost_dir.glob("*.ColocBoost.parquet")):
        if path.name.startswith("cross_"):
            continue
        tissue = path.name.replace(".ColocBoost.parquet", "")
        df = pd.read_parquet(path, columns=cols)
        df["tissue"] = tissue
        frames.append(df)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def build_cross_tissue_members(cross_tissue_loci_path, colocboost_dir, *, include_ucos=False):
    loci = pd.read_parquet(cross_tissue_loci_path)
    all_cos = load_main_colocboost(colocboost_dir).set_index("cos_id", drop=False)
    memb = loci.explode("member_cos_ids").rename(columns={"member_cos_ids": "cos_id"})
    memb["cos_id"] = memb["cos_id"].astype(str)
    cos_rows = memb[["locus_id", "cos_id"]].merge(
        all_cos.reset_index(drop=True), on="cos_id", how="inner"
    )
    cos_rows["lead_variant_id"] = [
        lead_from_lists(v, p) for v, p in zip(cos_rows["variants"], cos_rows["vcp"])
    ]
    if "phenotype_id" not in cos_rows.columns:
        cos_rows["phenotype_id"] = cos_rows["phenotype_ids"]

    qtl_by_locus = (
        cos_rows.groupby("locus_id")["gene_types_robust"]
        .agg(lambda s: _locus_qtl_label(_member_qtl_class(x) for x in s))
    )
    locus_summary = loci.copy()
    locus_summary["qtl_label"] = locus_summary["locus_id"].map(qtl_by_locus)

    if not include_ucos:
        return locus_summary, cos_rows

    member_ids = set(memb["cos_id"])
    has_gwas = all_cos["gwas_traits"].map(_coloc_list_len) > 0
    has_qtl = all_cos["gene_ids_robust"].map(_coloc_list_len) > 0
    ucos = all_cos.loc[
        ~all_cos.index.isin(member_ids) & (has_gwas ^ has_qtl)
    ].reset_index(drop=True)
    ucos = ucos.loc[has_gwas.reindex(ucos["cos_id"]).fillna(False).values].copy()
    ucos["locus_id"] = "ucos_" + ucos["cos_id"].astype(str)
    ucos["lead_variant_id"] = [
        lead_from_lists(v, p) for v, p in zip(ucos["variants"], ucos["vcp"])
    ]
    if "phenotype_id" not in ucos.columns:
        ucos["phenotype_id"] = ucos["phenotype_ids"]
    members = pd.concat([cos_rows, ucos], ignore_index=True)
    return locus_summary, members



## Paths

In [ ]:
ldsc_summary_path = Path("path/to/ldsc_summary.tsv")
trait_abbr_tsv = Path("path/to/table_s2_abbreviations.txt")
gwas_meta_path = Path("path/to/barbeira_full_gwas_metadata.txt")

gencode_class_path = Path("path/to/gencode_v39_v47_gene_classification.txt")
lnc_cat_path = Path("path/to/lncrna_category.txt")
gtex_median_tpm_path = Path("path/to/GTEx_Analysis_2025-08-22_v11_RNASeQCv2.4.3_gene_median_tpm.gct.gz")
tissue_names_file = Path("annotation/refs/tissue_name.txt")

eqtl_dir = Path("path/to/v11_eqtl_susie")
sqtl_dir = Path("path/to/v11_sqtl_susie")

colocboost_dir = Path("path/to")
loci_path = Path("path/to/cross_tissue_loci.parquet")

fig_dir = Path("figures")
fig_pdf = fig_dir / "figure_4.pdf"
fig_png = fig_dir / "figure_4.png"


## Load data


### Trait abbreviations


In [ ]:
s2_abbr = set()
with open(trait_abbr_tsv) as fh:
    fh.readline()
    for line in fh:
        parts = line.strip().split()
        if parts:
            s2_abbr.add(parts[-2])
print(f"Table S2 abbreviations: {len(s2_abbr)}")

### GWAS metadata / trait labels


In [ ]:
gwas_meta = pd.read_csv(gwas_meta_path, sep="\t")
trait_labels = {}
for tag, code in zip(gwas_meta["Tag"], gwas_meta["new_abbreviation"]):
    if pd.isna(code) or code not in s2_abbr:
        continue
    trait_labels[str(tag)] = str(code)
print(f"Table S2 trait labels: {len(trait_labels)}")

### LDSC primary summaries


In [ ]:
def prep_primary_summary(path):
    sub = pd.read_csv(path, sep="\t")
    sub = sub.dropna(
        subset=["Enrichment", "Enrichment_std_error", "tau_star", "tau_star_se", "tau_star_z", "h2"]
    ).copy()
    return sub[sub["Enrichment_std_error"] > 0].copy()


ldsc_df = prep_primary_summary(ldsc_summary_path)
print(f"LDSC summary: {len(ldsc_df):,} rows, {ldsc_df['model_id'].nunique()} models from {ldsc_summary_path.name}")

### GENCODE lncRNA classification


In [ ]:
gencode_genes = pd.read_table(gencode_class_path, low_memory=False)
gencode_genes = gencode_genes[gencode_genes["gene_type_grouped"] != "artifact"]
new_lncrna_ids = set(gencode_genes.loc[gencode_genes["gene_type_grouped"] == "novel_lncRNA", "gene_id_stable"])
existing_lncrna_ids = set(gencode_genes.loc[gencode_genes["gene_type_grouped"] == "existing_lncRNA", "gene_id_stable"])
protein_coding_ids = set(gencode_genes.loc[gencode_genes["gene_type_grouped"] == "protein_coding", "gene_id_stable"])
lncrna_ids = new_lncrna_ids | existing_lncrna_ids
stable_to_v47 = gencode_genes.drop_duplicates("gene_id_stable").set_index("gene_id_stable")["gene_id_v47"]
print(
    f"Novel lncRNAs: {len(new_lncrna_ids):,}; existing lncRNAs: {len(existing_lncrna_ids):,}; "
    f"protein-coding: {len(protein_coding_ids):,}"
)

### ColocBoost cross-tissue loci (summary and members)


In [ ]:
locus_summary, locus_long = build_cross_tissue_members(loci_path, colocboost_dir, include_ucos=False)
locus_summary["n_gwas_gws"] = locus_summary["gwas_gws_traits"].map(len)
locus_summary["qtl_label_raw"] = locus_summary["qtl_label"].astype(str)
locus_summary["qtl_label"] = (
    locus_summary["qtl_label_raw"].map(qtl_label_for_figure).fillna(locus_summary["qtl_label_raw"])
)
print(
    f"Loci: {len(locus_summary):,}; members: {len(locus_long):,}; "
    f"GWS: {(locus_summary['n_gwas_gws'] > 0).sum():,}"
)
print("figure qtl_label (weak → Shared):")
print(locus_summary["qtl_label"].value_counts().to_string())


### GTEx v11 gene median TPM

In [ ]:
def load_gtex_portal_median_tpm(path: Path) -> pd.DataFrame:
    """Load portal gene_median_tpm GCT; columns = pretty SMTSD names."""
    with gzip.open(path, "rt") as fh:
        fh.readline()
        fh.readline()
        raw = pd.read_csv(fh, sep="\t")
    med = raw.set_index("Name").drop(columns=["Description"], errors="ignore")
    tissue_map = pd.read_csv(tissue_names_file, sep="\t")
    stem_to_pretty = dict(zip(tissue_map["tissue_stem"], tissue_map["tissue_name"]))
    renamed = {}
    for c in med.columns:
        pretty = stem_to_pretty.get(c) or stem_to_pretty.get(c.replace("-", "_"))
        if pretty is None:
            raise KeyError(f"Cannot map portal median tissue column {c!r} to SMTSD")
        renamed[c] = pretty
    return med.rename(columns=renamed)


median_tpm = load_gtex_portal_median_tpm(gtex_median_tpm_path)
_before = median_tpm.shape[1]
# Max-TPM covariate uses 54 bulk sites (subtypes out; cell lines kept — not the τ set)
median_tpm = median_tpm[[c for c in median_tpm.columns if c not in histology_subtypes]]
if median_tpm.shape[1] != expected_n_bulk_tissues:
    raise ValueError(
        f"Expected {expected_n_bulk_tissues} bulk tissues after dropping histology subtypes, "
        f"found {median_tpm.shape[1]} (from {_before} portal columns)"
    )
max_tpm_by_v47 = median_tpm.max(axis=1)
print(
    f"Official median TPM: {median_tpm.shape[0]:,} genes × {median_tpm.shape[1]} bulk tissues "
    f"(dropped {_before - median_tpm.shape[1]} histology subtypes from {_before} portal columns)"
)

### lncRNA category annotations


In [ ]:
lnc_cat_all = pd.read_table(
    lnc_cat_path,
    usecols=[
        "gene_id", "gene_id_stable", "is_mirna_host", "is_ncORF",
        "expressed_dgtex", "expressed_macaque", "expressed_marmoset",
        "primate_cons_exon_unique", "primate_cons_sj_unique",
        "tss_e2g_enh",
        "is_tissue_specific", "peak_tissue",
        "macaque_tissue_concordant", "marmoset_tissue_concordant",
    ],
).drop_duplicates("gene_id_stable")
lnc_cat_all = lnc_cat_all[lnc_cat_all["gene_id_stable"].isin(lncrna_ids)].copy()
lnc_by_stable = lnc_cat_all.set_index("gene_id_stable")
print(f"lncRNA category rows in universe: {len(lnc_cat_all):,}")


### GTEx v11 eGenes (tissues tested)

In [ ]:
# Ascertainment covariate for panel-e ORs: count tissues where each gene appears in eGenes.
egene_files = sorted(eqtl_dir.glob("*.v11.eGenes.txt.gz"))
_egene_rows = []
for fpath in tqdm(egene_files, desc="eGenes tissues"):
    tissue = fpath.name.split(".v11.")[0]
    genes = pd.read_csv(fpath, sep="\t", usecols=["gene_id"])
    genes["gene_id_stable"] = genes["gene_id"].astype(str).str.split(".").str[0]
    genes["tissue"] = tissue
    _egene_rows.append(genes[["gene_id_stable", "tissue"]])
_egene_gt = pd.concat(_egene_rows, ignore_index=True)
n_eqtl_tested_by_stable = _egene_gt.groupby("gene_id_stable")["tissue"].nunique()
del _egene_rows, _egene_gt
print(f"eGenes files: {len(egene_files)}; genes with tissues-tested: {len(n_eqtl_tested_by_stable):,}")

### GTEx v11 sGenes (tissues tested)

In [ ]:
# Ascertainment covariate for panel-e ORs: count tissues where each gene appears in sGenes.
sgene_files = sorted(sqtl_dir.glob("*.v11.sGenes.txt.gz"))
_sgene_rows = []
for fpath in tqdm(sgene_files, desc="sGenes tissues"):
    tissue = fpath.name.split(".v11.")[0]
    genes = pd.read_csv(fpath, sep="\t", usecols=["gene_id"])
    genes["gene_id_stable"] = genes["gene_id"].astype(str).str.split(".").str[0]
    genes["tissue"] = tissue
    _sgene_rows.append(genes[["gene_id_stable", "tissue"]].drop_duplicates())
_sgene_gt = pd.concat(_sgene_rows, ignore_index=True)
n_sqtl_tested_by_stable = _sgene_gt.groupby("gene_id_stable")["tissue"].nunique()
del _sgene_rows, _sgene_gt
print(f"sGenes files: {len(sgene_files)}; genes with tissues-tested: {len(n_sqtl_tested_by_stable):,}")

## Run analysis


### Panel e annotation flags

In [ ]:
# Gene-level annotation flags used by panel-e OR traits.
# expressed_macaque/marmoset already require an ortholog id (classify_lncrna).
dgtex_by_stable = lnc_by_stable["expressed_dgtex"].astype(bool)
primate_expr_by_stable = (
    lnc_by_stable["expressed_macaque"].astype(bool)
    | lnc_by_stable["expressed_marmoset"].astype(bool)
)

nontestis_ok_by_stable = ~lnc_by_stable["peak_tissue"].eq("Testis")
mac_tc_by_stable = lnc_by_stable["macaque_tissue_concordant"].astype(bool)
mar_tc_by_stable = lnc_by_stable["marmoset_tissue_concordant"].astype(bool)

ts_by_stable = lnc_by_stable["is_tissue_specific"].astype(bool)
testis_peak_by_stable = lnc_by_stable["peak_tissue"].eq("Testis")
ts_testis_by_stable = ts_by_stable & testis_peak_by_stable
ts_nontestis_by_stable = ts_by_stable & nontestis_ok_by_stable
_nhp_match = mac_tc_by_stable | mar_tc_by_stable
tc_testis_by_stable = ts_testis_by_stable & _nhp_match
tc_nontestis_either_by_stable = ts_nontestis_by_stable & _nhp_match

max_tpm_by_stable = (
    lnc_by_stable.index.to_series()
    .map(stable_to_v47)
    .map(max_tpm_by_v47)
    .fillna(0)
    .astype(float)
)
tpm_bin_lt_1_by_stable = max_tpm_by_stable < 1.0
tpm_bin_gt_1_by_stable = max_tpm_by_stable >= 1.0

primate_cons_exon_unique_by_stable = lnc_by_stable["primate_cons_exon_unique"].astype(bool)
primate_cons_sj_unique_by_stable = lnc_by_stable["primate_cons_sj_unique"].astype(bool)
tss_e2g_enh_by_stable = lnc_by_stable["tss_e2g_enh"].astype(bool)

primate_expr_nontestis_by_stable = primate_expr_by_stable & nontestis_ok_by_stable

or_universe_ids = set(lncrna_ids)
print(f"OR universe (all annotated lncRNAs): {len(or_universe_ids):,}")
print(
    f"TPM bins <1 / >=1: {int(tpm_bin_lt_1_by_stable.sum()):,}/{int(tpm_bin_gt_1_by_stable.sum()):,}; "
    f"TS testis/non-testis: {int(ts_testis_by_stable.sum()):,}/{int(ts_nontestis_by_stable.sum()):,}; "
    f"TC testis/non-testis: {int(tc_testis_by_stable.sum()):,}/{int(tc_nontestis_either_by_stable.sum()):,}; "
    f"primate expr (non-Testis peak): {int(primate_expr_nontestis_by_stable.sum()):,}"
)


### Trait cohort and merged LDSC table


In [ ]:
trait_cohort = np.array(sorted(set(trait_labels) & set(ldsc_df["trait_id"])))
n_traits = len(trait_cohort)

df = ldsc_df[ldsc_df["trait_id"].isin(trait_cohort)].copy()
df = df.drop_duplicates(subset=["trait_id", "model_id"], keep="last")

print(f"Trait cohort: {n_traits} traits")
print(f"LDSC table: {len(df):,} rows × {df['model_id'].nunique()} models")

### Model IDs and IVW meta-analysis (panels a–b)


In [ ]:
panel_b_scopes = ("base_only", "base_pc_esqtl", "marginal")
panel_b_scopes_no_joint = ("base_only", "marginal")


def model_id(biotype, scope, q):
    stem = ds_rep if biotype == "protein_coding_downsampled" else biotype
    if scope == "base_only":
        return f"single_{stem}_{q}{biotype_model_suffix}_baseOnly"
    if scope == "base_pc_esqtl":
        if biotype == "protein_coding":
            if q == "sqtl":
                return f"single_protein_coding_esqtl{biotype_model_suffix}_baseOnly_focal_sqtl"
            return f"single_protein_coding_esqtl{biotype_model_suffix}_baseOnly"
        return f"joint_{stem}_vs_protein_coding_esqtl{biotype_model_suffix}_baseOnly_focal_{q}"
    return f"single_{stem}_{q}{biotype_model_suffix}"


def panel_b_scopes_for(biotype):
    """PC / d.s. PC: QTL and QTL | LDSC baseline; lncRNAs (all/existing/novel) also QTL | p.c. e/sQTL."""
    if biotype in ("protein_coding", "protein_coding_downsampled"):
        return panel_b_scopes_no_joint
    return panel_b_scopes


expected_models = [
    model_id(biotype, scope, q)
    for biotype, _ in biotype_groups
    for q in ("eqtl", "sqtl")
    for scope in panel_b_scopes_for(biotype)
]
# Novel lncRNA sQTL is intentionally absent (sparse; plotted as such in panel a/b).
known_sparse_models = {
    model_id("novel_lncRNA", scope, "sqtl")
    for scope in panel_b_scopes_for("novel_lncRNA")
}
missing_models = sorted(set(expected_models) - set(df["model_id"]))
unexpected_missing = [m for m in missing_models if m not in known_sparse_models]
if unexpected_missing:
    raise KeyError(
        "Missing panel B LDSC model IDs (run LDSC / re-parse before plotting):\n  "
        + "\n  ".join(unexpected_missing)
    )
sparse_missing = [m for m in missing_models if m in known_sparse_models]
if sparse_missing:
    print(f"Panel B: {len(sparse_missing)} novel lncRNA sQTL models absent (expected sparse)")
print(
    f"Panel B grid: {len(expected_models) - len(sparse_missing)}/{len(expected_models)} "
    f"non-sparse model IDs present"
)


def ivw_meta(sub, vcol="Enrichment", scol="Enrichment_std_error", null=1.0, one_sided_gt=False):
    y = pd.to_numeric(sub[vcol], errors="coerce").to_numpy(float)
    se = pd.to_numeric(sub[scol], errors="coerce").to_numpy(float)
    ok = np.isfinite(y) & np.isfinite(se) & (se > 0)
    y, se = y[ok], se[ok]
    if len(y) == 0:
        return None
    w = 1.0 / se**2
    mu = float(np.dot(w, y) / w.sum())
    if len(y) > 1:
        sig2 = max(float(np.dot(w, (y - mu) ** 2) / (len(y) - 1)), 0.0)
        semu = sig2**0.5 / w.sum() ** 0.5
    else:
        semu = float(se[0])
    z1 = (mu - null) / semu if semu > 0 else np.nan
    if one_sided_gt:
        p1 = float(stats.norm.sf(z1)) if np.isfinite(z1) else np.nan
    else:
        p1 = float(2 * stats.norm.sf(abs(z1))) if np.isfinite(z1) else np.nan
    return dict(
        n=len(y),
        mu=mu,
        se=semu,
        z=z1,
        ci_lo=mu - 1.96 * semu,
        ci_hi=mu + 1.96 * semu,
        p_vs1=p1 if null == 1.0 and not one_sided_gt else np.nan,
        p_vs0=p1 if null == 0.0 and not one_sided_gt else np.nan,
        p_gt0=p1 if null == 0.0 and one_sided_gt else np.nan,
        p=p1,
    )


def meta(mid):
    return ivw_meta(df[df["model_id"] == mid])


def meta_tau_star(mid):
    sub = df[df["model_id"] == mid].copy()
    sub = sub.dropna(subset=["tau_star", "tau_star_se", "h2"])
    sub = sub[(sub["tau_star_se"] > 0) & (sub["h2"] > 0)].copy()
    if sub.empty:
        return None
    for col in ("tau_star", "tau_star_se", "tau_star_z"):
        sub[col] = pd.to_numeric(sub[col], errors="coerce")
    return ivw_meta(sub, "tau_star", "tau_star_se", null=0.0, one_sided_gt=True)


print("Meta-analysis helpers ready (enrichment null=1; τ* null=0, one-sided)")

### UpSet, gene CoS counts, and lncRNA-only vs shared CoS-class ORs (panels c–e)


In [ ]:
def as_stem_set(vals):
    out = set()
    if vals is None:
        return out
    try:
        if isinstance(vals, float) and np.isnan(vals):
            return out
    except (TypeError, ValueError):
        pass
    items = vals if isinstance(vals, (list, tuple, np.ndarray, set)) else [vals]
    for v in items:
        if v is None:
            continue
        m = ensg_stem.search(str(v))
        if m:
            out.add(m.group(1))
    return out

def as_str_list(vals):
    if vals is None:
        return []
    try:
        if isinstance(vals, float) and np.isnan(vals):
            return []
    except (TypeError, ValueError):
        pass
    if isinstance(vals, (list, tuple, np.ndarray, set)):
        return [str(v) for v in vals if v is not None and str(v) not in ("", "nan")]
    return [str(vals)] if str(vals) not in ("", "nan") else []

def parse_neglog10_list(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return []
    out = []
    for tok in str(val).split(","):
        tok = tok.strip()
        if not tok or tok.lower() == "nan":
            out.append(np.nan)
        else:
            try:
                out.append(float(tok))
            except ValueError:
                out.append(np.nan)
    return out

def has_gwas_at_cut(phenotype_id, max_neg_log10_p, cut=gwas_neglog10p_1e5) -> bool:
    """True if any GWAS phenotype tip has CS-max -log10(p) > cut."""
    parts = [p.strip() for p in str(phenotype_id).split(";") if p.strip()]
    vals = parse_neglog10_list(max_neg_log10_p)
    for i, part in enumerate(parts):
        if i >= len(vals):
            break
        pl = part.lower()
        if "expression" in pl or "splicing" in pl:
            continue
        v = vals[i]
        if np.isfinite(v) and float(v) > cut:
            return True
    return False

def prepare_gene_coloc_loci(df, name):
    """CoS loci with QTL label; drop no_qtl if present."""
    out = df.copy()
    n_no_qtl = int((out["qtl_label"].astype(str) == "no_qtl").sum())
    if n_no_qtl:
        print(f"WARN: dropping {n_no_qtl:,} {name} loci with qtl_label=no_qtl")
        out = out[out["qtl_label"].astype(str) != "no_qtl"].copy()
    out["gene_stems"] = out["gene_ids_robust"].map(as_stem_set)
    print(f"{name} gene-coloc loci: {len(out):,}")
    return out

# Loci with any member at CS-max GWAS p < 1e-5; GWS loci (p < 5.7e-9) from n_gwas_gws > 0
_member_1e5 = [
    has_gwas_at_cut(pid, csmax, cut=gwas_neglog10p_1e5)
    for pid, csmax in zip(locus_long["phenotype_ids"], locus_long["cos_max_pval_nominal_phenotypes"])
]
locus_long_1e5 = locus_long[np.asarray(_member_1e5, dtype=bool)].copy()
loci_1e5_ids = set(locus_long_1e5["locus_id"].astype(str))
print(
    f"p-tier filter: members all={len(locus_long):,} → p<1e-5={len(locus_long_1e5):,}; "
    f"loci all={len(locus_summary):,} → p<1e-5={sum(locus_summary['locus_id'].astype(str).isin(loci_1e5_ids)):,}; "
    f"GWS={int((locus_summary['n_gwas_gws'] > 0).sum()):,}"
)
all_loci = prepare_gene_coloc_loci(
    locus_summary[locus_summary["locus_id"].astype(str).isin(loci_1e5_ids)],
    "p<1e-5",
)
gws_loci = prepare_gene_coloc_loci(
    locus_summary[locus_summary["n_gwas_gws"] > 0], "GWS"
)

# UpSet biotypes: protein-coding, existing lncRNA, novel lncRNA, pseudogene
upset_biotypes = ["protein_coding", "existing_lncRNA", "novel_lncRNA", "pseudogene"]
upset_set_labels = {
    "protein_coding": "Protein-coding",
    "existing_lncRNA": "Existing lncRNA",
    "novel_lncRNA": "Novel lncRNA",
    "pseudogene": "Pseudogene",
}

def locus_biotype_combo_for(types, biotypes):
    """Map QTL types → ordered biotype tuple (only `biotypes` members)."""
    present = set(as_str_list(types))
    return tuple(bt for bt in biotypes if bt in present)

def upset_counts_for(loci_df, biotypes=None, combo_fn=None):
    biotypes = upset_biotypes if biotypes is None else list(biotypes)
    if combo_fn is None:
        combo_fn = lambda types, _bt=biotypes: locus_biotype_combo_for(types, _bt)
    tmp = loci_df.copy()
    tmp["biotype_combo"] = tmp["gene_types_robust"].map(combo_fn)
    counts = (
        tmp["biotype_combo"]
        .value_counts()
        .rename_axis("combo")
        .reset_index(name="n")
    )
    counts = counts[counts["combo"].map(len) > 0].copy()
    return tmp, counts

def build_upset_tables(loci_1e5_df, loci_gws_df, biotypes, set_labels):
    """Intersection counts + set sizes for one UpSet biotype universe."""
    all_u, upset_all = upset_counts_for(loci_1e5_df, biotypes=biotypes)
    gws_u, upset_gws = upset_counts_for(loci_gws_df, biotypes=biotypes)
    combo_counts = (
        upset_all.rename(columns={"n": "n_all"})
        .merge(upset_gws.rename(columns={"n": "n_gws"}), on="combo", how="outer")
        .fillna({"n_all": 0, "n_gws": 0})
    )
    combo_counts["n_all"] = combo_counts["n_all"].astype(int)
    combo_counts["n_gws"] = combo_counts["n_gws"].astype(int)
    combo_counts["n"] = combo_counts[["n_all", "n_gws"]].max(axis=1)
    combo_counts = combo_counts.sort_values("n_all", ascending=False).reset_index(drop=True)
    set_sizes = {
        bt: int(all_u["biotype_combo"].map(lambda c: bt in c).sum()) for bt in biotypes
    }
    set_sizes_gws = {
        bt: int(gws_u["biotype_combo"].map(lambda c: bt in c).sum()) for bt in biotypes
    }
    return {
        "all_loci": all_u,
        "gws_loci": gws_u,
        "upset_combo_counts": combo_counts,
        "upset_set_sizes": set_sizes,
        "upset_set_sizes_gws": set_sizes_gws,
        "biotypes": list(biotypes),
        "set_labels": dict(set_labels),
    }

# UpSet v1 tables
_upset_v1 = build_upset_tables(all_loci, gws_loci, upset_biotypes, upset_set_labels)
all_loci = _upset_v1["all_loci"]
gws_loci = _upset_v1["gws_loci"]
upset_combo_counts = _upset_v1["upset_combo_counts"]
upset_set_sizes = _upset_v1["upset_set_sizes"]
upset_set_sizes_gws = _upset_v1["upset_set_sizes_gws"]
print("UpSet biotype intersections (p<1e-5 / GWS gene-coloc loci):")
print(upset_combo_counts.to_string(index=False))
print("UpSet set sizes (p<1e-5):", upset_set_sizes)
print("UpSet set sizes (GWS):", upset_set_sizes_gws)

def lnc_genes_in_qtl_label(loci_df, label, biotype_ids):
    genes = set()
    sub = loci_df[loci_df["qtl_label"].astype(str) == label]
    for r in sub.itertuples(index=False):
        genes |= set(r.gene_stems) & biotype_ids
    return genes

def coloc_lnc_genes(loci_df):
    return (
        lnc_genes_in_qtl_label(loci_df, "Shared", all_lncrna_ids)
        | lnc_genes_in_qtl_label(loci_df, "lncRNA-specific", all_lncrna_ids)
    )

all_lncrna_ids = new_lncrna_ids | existing_lncrna_ids
all_coloc_lnc_all = coloc_lnc_genes(all_loci)
all_coloc_lnc_gws = coloc_lnc_genes(gws_loci)

# Shared-with-non-lncRNA CoS membership (comparison group for panel-e ORs)
share_shared = "shared"
shared_genes_1e5 = lnc_genes_in_qtl_label(all_loci, "Shared", all_lncrna_ids)
print(
    f"  lncRNA coloc genes (p<1e-5): {len(all_coloc_lnc_all):,}; "
    f"GWS: {len(all_coloc_lnc_gws):,}"
)
print(
    f"  shared-with-non-lncRNA CoS genes: {len(shared_genes_1e5):,}"
)

def lnc_genes_matching_combo(loci_df, biotype_ids, pred):
    genes = set()
    for r in loci_df.itertuples(index=False):
        if not pred(r.biotype_combo):
            continue
        genes |= set(r.gene_stems) & biotype_ids
    return genes

# Panel d solids / panel e OR positives: exact lncRNA-only CoS combos (no PC, no pseudo)
# novel bar = novel-only; existing bar = existing-only + novel+existing
novel_only_combos = {("novel_lncRNA",)}
existing_only_combos = {("existing_lncRNA",)}
novel_existing_combos = {("existing_lncRNA", "novel_lncRNA")}
lnc_only_exact_combos = novel_only_combos | existing_only_combos | novel_existing_combos

genes_novel_only = lnc_genes_matching_combo(
    all_loci, new_lncrna_ids, lambda c: tuple(c) in novel_only_combos
)
genes_existing_only = lnc_genes_matching_combo(
    all_loci, existing_lncrna_ids, lambda c: tuple(c) in existing_only_combos
)
genes_existing_joint = lnc_genes_matching_combo(
    all_loci, existing_lncrna_ids, lambda c: tuple(c) in novel_existing_combos
)
# Existing lncRNA-only solid = existing-only ∪ novel+existing (same color in d)
genes_existing_lnc_only = genes_existing_only | genes_existing_joint
# Other CoS = shared-with-non-lncRNA, excluding genes already in the colored solid
genes_novel_other = (shared_genes_1e5 & new_lncrna_ids) - genes_novel_only
genes_existing_other = (shared_genes_1e5 & existing_lncrna_ids) - genes_existing_lnc_only
specific_genes_lnc_only = lnc_genes_matching_combo(
    all_loci, all_lncrna_ids, lambda c: tuple(c) in lnc_only_exact_combos
)
print(
    f"  panel-d novel: only={len(genes_novel_only):,}; other={len(genes_novel_other):,}"
)
print(
    f"  panel-d existing: lncRNA-only={len(genes_existing_lnc_only):,} "
    f"(existing-only={len(genes_existing_only):,}; novel+existing={len(genes_existing_joint):,}); "
    f"other={len(genes_existing_other):,}"
)
print(f"  OR lncRNA-only positives={len(specific_genes_lnc_only):,}")

share_lnc_only = "lncRNA-only"
sharing_group_specs = [
    (share_lnc_only, all_lncrna_ids, specific_genes_lnc_only),
    (share_shared, all_lncrna_ids, shared_genes_1e5),
]
panel_e_group_order = [share_lnc_only, share_shared]
print(
    f"  CoS class (lncRNA-only exact): n_specific={len(specific_genes_lnc_only):,}; "
    f"shared-with-non-lncRNA={len(shared_genes_1e5):,}; "
    f"in both={len(specific_genes_lnc_only & shared_genes_1e5):,}"
)

# Panel d: unique coloc genes — novel / existing only
gene_cos_rows = [
    ("novel_lncRNA", "novel\nlncRNA", new_lncrna_ids),
    ("existing_lncRNA", "existing\nlncRNA", existing_lncrna_ids),
]

def _overall_coloc_counts(rows):
    out = {}
    for key, lab, ids in rows:
        overall = set()
        for r in all_loci.itertuples(index=False):
            overall |= set(r.gene_stems) & ids
        out[key] = len(overall)
        print(f"  {lab.replace(chr(10), ' ')}: overall={len(overall):,}")
    return out

print("Unique coloc genes (any CoS) and panel-d stacks:")
_overall = _overall_coloc_counts(gene_cos_rows)
gene_cos_counts = {
    "novel_lncRNA": {
        "overall": _overall["novel_lncRNA"],
        "only": len(genes_novel_only),
        "other": len(genes_novel_other),
    },
    "existing_lncRNA": {
        "overall": _overall["existing_lncRNA"],
        "only": len(genes_existing_lnc_only),  # existing-only ∪ novel+existing
        "other": len(genes_existing_other),
    },
}
for key, lab in [("novel_lncRNA", "novel"), ("existing_lncRNA", "existing")]:
    s = gene_cos_counts[key]
    print(
        f"  {lab}: any={s['overall']:,}; "
        f"colored={s['only']:,} + other={s['other']:,} "
        f"(stack={s['only'] + s['other']:,})"
    )

# Categories: enhancer–seq; primate syn / either-NHP expr; dGTEx; TS/TC × testis last
or_traits = [
    ("tss_e2g_enh", "enhancer", None),
    ("is_mirna_host", "miRNA\nhost", None),
    ("is_ncORF", "translated\nncORF", None),
    # unique-to-gene Zoonomia constraint (Sullivan); labels omit "unique"
    ("primate_cons_exon_unique", "primate\nexon\ncons.", None),
    ("primate_cons_sj_unique", "primate\nsplice junct.\ncons.", None),
    # either-NHP expr (macaque|marmoset, non-testis) and dGTEx as overlapping binaries
    ("primate_expr", "primate\nexpr.", None),
    ("dev_expr", "dev.\nexpr.", None),
    # mutually exclusive TS/TC × testis levels; ref = not T.S. (omitted)
    # TS = tissue-specific without NHP organ match; TC = tissue-conserved (NHP match)
    # individual: each vs not-T.S. (exclude other exclusive levels via subset)
    ("ts_testis", "TS\n(testis)", "ts_vs_not_ts_testis"),
    ("ts_nontestis_only", "TS\n(non-testis)", "ts_vs_not_ts_nontestis"),
    ("tc_testis", "TC\n(testis)", "ts_vs_not_tc_testis"),
    ("tc_nontestis", "TC\n(non-testis)", "ts_vs_not_tc_nontestis"),
]

def build_trait_gld(coloc_gene_ids, biotype_ids, universe_ids=None):
    base_universe = or_universe_ids if universe_ids is None else universe_ids
    universe = base_universe & biotype_ids
    gld = lnc_cat_all[lnc_cat_all["gene_id_stable"].isin(universe)].copy()
    gld["has_coloc"] = gld["gene_id_stable"].isin(coloc_gene_ids).astype(int)
    gene_v47 = gld["gene_id_stable"].map(stable_to_v47)
    gld["log_max_tpm"] = np.log10(gene_v47.map(max_tpm_by_v47).fillna(0) + 1)
    gld["tss_e2g_enh"] = gld["gene_id_stable"].map(tss_e2g_enh_by_stable).fillna(False).astype(int)
    gld["is_mirna_host"] = gld["is_mirna_host"].astype(int)
    gld["is_ncORF"] = gld["is_ncORF"].astype(int)
    gld["primate_cons_exon_unique"] = gld["gene_id_stable"].map(primate_cons_exon_unique_by_stable).fillna(False).astype(int)
    gld["primate_cons_sj_unique"] = gld["gene_id_stable"].map(primate_cons_sj_unique_by_stable).fillna(False).astype(int)
    # mutually exclusive TS/TC × testis levels (not T.S. = reference)
    _ts_testis = gld["gene_id_stable"].map(ts_testis_by_stable).fillna(False).astype(int)
    _ts_nontestis = gld["gene_id_stable"].map(ts_nontestis_by_stable).fillna(False).astype(int)
    gld["tc_testis"] = gld["gene_id_stable"].map(tc_testis_by_stable).fillna(False).astype(int)
    gld["tc_nontestis"] = gld["gene_id_stable"].map(tc_nontestis_either_by_stable).fillna(False).astype(int)
    gld["ts_testis"] = ((_ts_testis == 1) & (gld["tc_testis"] == 0)).astype(int)
    gld["ts_nontestis_only"] = ((_ts_nontestis == 1) & (gld["tc_nontestis"] == 0)).astype(int)
    # individual subsets: not-T.S. ∪ focal level (exclude other exclusive TS/TC levels)
    gld["ts_vs_not_ts_testis"] = (
        (gld["ts_nontestis_only"] == 0) & (gld["tc_testis"] == 0) & (gld["tc_nontestis"] == 0)
    ).astype(int)
    gld["ts_vs_not_ts_nontestis"] = (
        (gld["ts_testis"] == 0) & (gld["tc_testis"] == 0) & (gld["tc_nontestis"] == 0)
    ).astype(int)
    gld["ts_vs_not_tc_testis"] = (
        (gld["ts_testis"] == 0) & (gld["ts_nontestis_only"] == 0) & (gld["tc_nontestis"] == 0)
    ).astype(int)
    gld["ts_vs_not_tc_nontestis"] = (
        (gld["ts_testis"] == 0) & (gld["ts_nontestis_only"] == 0) & (gld["tc_testis"] == 0)
    ).astype(int)
    # either-NHP expr (ortholog expressed macaque|marmoset, non-testis) and dGTEx
    gld["primate_expr"] = gld["gene_id_stable"].map(primate_expr_nontestis_by_stable).fillna(False).astype(int)
    gld["dev_expr"] = gld["gene_id_stable"].map(dgtex_by_stable).fillna(False).astype(int)
    gld["n_tissues_tested_eqtl"] = (
        gld["gene_id_stable"].map(n_eqtl_tested_by_stable).fillna(0).astype(float)
    )
    gld["n_tissues_tested_sqtl"] = (
        gld["gene_id_stable"].map(n_sqtl_tested_by_stable).fillna(0).astype(float)
    )
    return gld

def fit_trait_or(gld, trait_col, subset_col=None, covars=""):
    work = gld[gld[subset_col] == 1].copy() if subset_col is not None else gld.copy()
    if work.empty:
        return None
    n_coloc = int(work["has_coloc"].sum())
    n_non = len(work) - n_coloc
    if n_coloc < 3 or n_non < 3:
        return None
    if work["has_coloc"].nunique() < 2 or work[trait_col].nunique() < 2:
        return None
    # Binary traits: require both trait+/trait− among coloc genes
    is_binary = set(pd.unique(work[trait_col].dropna())).issubset({0, 1, 0.0, 1.0, True, False})
    if is_binary:
        coloc = work["has_coloc"] == 1
        n_coloc_trait = int(work.loc[coloc, trait_col].sum())
        n_coloc_notrait = n_coloc - n_coloc_trait
        if work[trait_col].sum() < 3:
            return None
        if n_coloc_trait == 0 or n_coloc_notrait == 0:
            return None
    covar_cols = [c.strip() for c in covars.split("+")] if covars else []
    for c in covar_cols:
        if c not in work.columns or work[c].nunique() < 2:
            return None
    formula = f"has_coloc ~ {trait_col}" + (f" + {covars}" if covars else "")
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            fit = smf.logit(formula, data=work).fit(disp=False, maxiter=200)
    except Exception:
        return None
    coef = float(fit.params[trait_col])
    se = float(fit.bse[trait_col])
    if not np.isfinite(coef) or not np.isfinite(se):
        return None
    ci = fit.conf_int().loc[trait_col]
    or_hat = float(np.exp(coef))
    ci_low = float(np.exp(ci.iloc[0]))
    ci_high = float(np.exp(ci.iloc[1]))
    if not np.isfinite(or_hat) or not np.isfinite(ci_low) or not np.isfinite(ci_high):
        return None
    return {
        "trait": trait_col,
        "OR": or_hat,
        "CI_low": ci_low,
        "CI_high": ci_high,
        "pvalue": float(fit.pvalues[trait_col]),
        "n_coloc": n_coloc,
    }

def compute_sharing_or_df(group_specs, scope, universe_ids, traits, covars):
    rows = []
    for group_name, biotype_ids, genes in group_specs:
        gld = build_trait_gld(genes, biotype_ids, universe_ids=universe_ids)
        for trait_col, trait_label, subset_col in traits:
            row = fit_trait_or(
                gld, trait_col, subset_col=subset_col, covars=covars,
            ) or {
                "trait": trait_col,
                "OR": np.nan,
                "CI_low": np.nan,
                "CI_high": np.nan,
                "pvalue": np.nan,
                "n_coloc": int(gld["has_coloc"].sum()) if subset_col is None else int(
                    gld.loc[gld[subset_col] == 1, "has_coloc"].sum()
                ),
            }
            row["label"] = trait_label
            row["group"] = group_name
            row["adjust_expr"] = bool(covars)
            row["covars"] = covars
            row["scope"] = scope
            rows.append(row)
    return pd.DataFrame(rows)

# Ascertainment covariates: expression + number of e/sQTL tissues tested
adj_covars = "log_max_tpm + n_tissues_tested_eqtl + n_tissues_tested_sqtl"

# Panel e: lncRNA-only vs shared CoS-class membership ORs (p<1e-5)
print("Panel e: lncRNA-only vs shared CoS class ORs (annotation + expr. + N tissues; p<1e-5)")
sharing_or_df = compute_sharing_or_df(
    sharing_group_specs,
    scope="p<1e-5 lncRNA-only",
    universe_ids=lncrna_ids,
    traits=or_traits,
    covars=adj_covars,
)
for class_name, _bio, class_ids in sharing_group_specs:
    _gld = build_trait_gld(class_ids, _bio, universe_ids=lncrna_ids)
    print(
        f"  {class_name}: n={len(_gld):,}, n_pos={int(_gld.has_coloc.sum()):,}, "
        f"adj covars: {adj_covars}"
    )

def or_cat_order_by_mean(or_df, traits_list, groups=None):
    groups = list(panel_e_group_order) if groups is None else list(groups)

    def _key(tls):
        trait_col = tls[0]
        ors = []
        for g in groups:
            sub = or_df[(or_df["trait"] == trait_col) & (or_df["group"] == g)]
            if len(sub) and np.isfinite(sub.iloc[0]["OR"]):
                ors.append(float(sub.iloc[0]["OR"]))
        if not ors:
            return (1, 0.0)
        return (0, -float(np.mean(ors)))

    traits_sorted = sorted(list(traits_list), key=_key)
    return (
        [lab for _t, lab, _s in traits_sorted],
        [t for t, _lab, _s in traits_sorted],
    )

panel_e_cat_order, panel_e_trait_order = or_cat_order_by_mean(
    sharing_or_df, or_traits
)
print("Panel e category order (by mean lncRNA-only+shared OR):", panel_e_trait_order)


## Make figure


In [ ]:
eqtl_handle = Line2D([], [], marker="o", ls="", markerfacecolor="grey", markeredgecolor="k",
                     markeredgewidth=0.35, ms=4, label="eQTL")
sqtl_handle = Line2D([], [], marker="s", ls="", markerfacecolor="grey", markeredgecolor="k",
                     markeredgewidth=0.35, ms=4, label="sQTL")
legend_a_handles = [eqtl_handle, sqtl_handle]
tau_qtl_leg = [
    Line2D([], [], marker="o", ls="", markerfacecolor="white", markeredgecolor="k", markeredgewidth=0.35, ms=4.5, label="QTL"),
    Line2D([], [], marker="o", ls="", markerfacecolor="#d9d9d9", markeredgecolor="k", markeredgewidth=0.35, ms=4.5, label="QTL | p.c. e/sQTL"),
    Line2D([], [], marker="o", ls="", markerfacecolor="grey", markeredgecolor="k", markeredgewidth=0.35, ms=4.5, label="QTL | LDSC baseline"),
]

def sig_stars(p):
    if p is None or not np.isfinite(p):
        return ""
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 0.05:
        return "*"
    return ""

modality_or_face = {
    share_lnc_only: "k",
    share_shared: "#BDBDBD",
}
modality_or_edge = {
    share_lnc_only: "k",
    share_shared: "k",
}
modality_or_colors = {
    share_lnc_only: "k",
    share_shared: "k",
}
modality_or_markers = {
    share_lnc_only: "o",
    share_shared: "o",
}

upset_combo_counts_i = upset_combo_counts
upset_set_sizes_i = upset_set_sizes
upset_set_sizes_gws_i = upset_set_sizes_gws
upset_biotypes_i = upset_biotypes
upset_set_labels_i = upset_set_labels
all_loci_i = all_loci
gws_loci_i = gws_loci
or_df_i = sharing_or_df
cat_order_i = panel_e_cat_order
or_groups_i = panel_e_group_order
or_xlabel_i = "OR gene w/ QTL-GWAS CoS"
gene_cos_rows_i = gene_cos_rows
gene_cos_counts_i = gene_cos_counts
gene_cos_spec_legend_i = "lncRNA-only CoS"

# Layout: left = a|b over c (UpSet); right = d over e
fig = plt.figure(figsize=(7.2, 5.6), dpi=300)
gs_main = fig.add_gridspec(1, 2, width_ratios=[2.15, 1.0], wspace=0.28)

gs_left = gs_main[0].subgridspec(2, 1, height_ratios=[1.50, 0.80], hspace=0.42)
gs_ab = gs_left[0].subgridspec(1, 2, width_ratios=[1, 1], wspace=0.18)
ax_enrich = fig.add_subplot(gs_ab[0, 0])
ax_tau = fig.add_subplot(gs_ab[0, 1], sharey=ax_enrich)

gs_f = gs_left[1].subgridspec(
    2, 2, height_ratios=[0.52, 0.48], width_ratios=[0.32, 0.68],
    hspace=0.08, wspace=0.62,
)
ax_e_leg = fig.add_subplot(gs_f[0, 0])
ax_e_leg.set_axis_off()
ax_e_bar = fig.add_subplot(gs_f[0, 1])
ax_e_dot = fig.add_subplot(gs_f[1, 1], sharex=ax_e_bar)
ax_e_set = fig.add_subplot(gs_f[1, 0], sharey=ax_e_dot)

gs_cd = gs_main[1].subgridspec(2, 1, height_ratios=[0.42, 1.58], hspace=0.28)
ax_genes = fig.add_subplot(gs_cd[0, 0])
ax_or = fig.add_subplot(gs_cd[1, 0])

# Shared data-unit step so equal-width a/b have matching physical dot spacing
point_step = 0.16
point_layout_lnc = (
    ("base_only", "eqtl", -2.5), ("base_only", "sqtl", -1.5),
    ("base_pc_esqtl", "eqtl", -0.5), ("base_pc_esqtl", "sqtl", 0.5),
    ("marginal", "eqtl", 1.5), ("marginal", "sqtl", 2.5),
)
point_layout_pc = (
    ("base_only", "eqtl", -1.5), ("base_only", "sqtl", -0.5),
    ("marginal", "eqtl", 0.5), ("marginal", "sqtl", 1.5),
)
biotype_ylabels = [lab for _, lab in biotype_groups]
n_bio_ab = len(biotype_groups)
all_tau_vals, label_pts = [], []

for gi, (biotype, _label) in enumerate(biotype_groups):
    edge = gene_type_palette[biotype]
    offsets = (np.arange(2) - 0.5) * point_step
    for q, dy, marker in zip(["eqtl", "sqtl"], offsets, ["o", "s"]):
        y = gi + dy
        if biotype == "novel_lncRNA" and q == "sqtl":
            ax_enrich.plot(
                1.0, y, marker="x", color="0.55", ms=2.4, mew=0.8, zorder=3,
            )
            continue
        m = meta(model_id(biotype, "marginal", q))
        if m is None:
            continue
        ax_enrich.errorbar(
            [m["mu"]], [y],
            xerr=[[m["mu"] - m["ci_lo"]], [m["ci_hi"] - m["mu"]]],
            fmt=marker, ms=2.4, color=edge, ecolor=edge,
            markerfacecolor=edge, markeredgecolor="k", markeredgewidth=0.3,
            capsize=0, lw=0.7, zorder=3,
        )
        sig = sig_stars(m["p_vs1"])
        if sig:
            ax_enrich.text(
                m["ci_hi"] + 0.08, y, sig,
                ha="left", va="center", color="black", fontsize=3.0, zorder=4,
            )

for _y in range(n_bio_ab):
    if _y % 2 == 1:
        ax_enrich.axhspan(_y - 0.5, _y + 0.5, color="#eeeeee", zorder=0, lw=0)
ax_enrich.axvline(1, color="#888888", ls=":", lw=0.8, zorder=1)
ax_enrich.set_yticks(range(n_bio_ab))
ax_enrich.set_yticklabels(biotype_ylabels, linespacing=0.75, fontsize=4.5)
ax_enrich.set_ylim(n_bio_ab - 0.4, -0.6)  # p.c. at top
ax_enrich.set_xlim(0.7, 4.0)
ax_enrich.set_xticks([0, 1, 2, 3, 4])
ax_enrich.set_xlabel("Heritability enrichment")
ax_enrich.tick_params(axis="y", length=0)
ax_enrich.spines[["top", "right", "left"]].set_visible(False)
ax_enrich.legend(
    handles=legend_a_handles, loc="lower center", bbox_to_anchor=(0.5, 1.0),
    frameon=False, borderaxespad=0.0, ncol=2, handletextpad=0.7, columnspacing=1.2,
    fontsize=3.5,
)

for gi, (biotype, _label) in enumerate(biotype_groups):
    edge = gene_type_palette[biotype]
    point_layout = (
        point_layout_pc if biotype in ("protein_coding", "protein_coding_downsampled")
        else point_layout_lnc
    )
    for scope, q, off_i in point_layout:
        y = gi + off_i * point_step
        if biotype == "novel_lncRNA" and q == "sqtl":
            ax_tau.plot(
                0.0, y, marker="x", color="0.55", ms=2.4, mew=0.8, zorder=3,
            )
            continue
        m = meta_tau_star(model_id(biotype, scope, q))
        if m is None:
            continue
        mfc = (
            "grey" if scope == "marginal"
            else ("#d9d9d9" if scope == "base_pc_esqtl" else "white")
        )
        ax_tau.errorbar(
            [m["mu"]], [y],
            xerr=[[m["mu"] - m["ci_lo"]], [m["ci_hi"] - m["mu"]]],
            fmt=qtl_marker[q], ms=2.4, color=edge, ecolor=edge,
            markerfacecolor=mfc, markeredgecolor=edge, markeredgewidth=0.3,
            capsize=0, lw=0.7, zorder=2,
        )
        all_tau_vals.extend([m["ci_hi"], m["ci_lo"]])
        label_pts.append((m["ci_hi"], y, sig_stars(m["p_gt0"]), edge))

for _y in range(n_bio_ab):
    if _y % 2 == 1:
        ax_tau.axhspan(_y - 0.5, _y + 0.5, color="#eeeeee", zorder=0, lw=0)
ax_tau.axvline(0.0, color="#888888", ls=":", lw=0.8, zorder=1)
tau_lo = min(all_tau_vals) if all_tau_vals else -0.2
tau_hi = max(all_tau_vals) if all_tau_vals else 1.0
pad = 0.12 * (tau_hi - tau_lo if tau_hi > tau_lo else 1.0)
ax_tau.set_xlim(tau_lo - pad, tau_hi + pad)
ax_tau.set_yticks(range(n_bio_ab))
# shared y with panel a; hide duplicate biotype labels
plt.setp(ax_tau.get_yticklabels(), visible=False)
for xhi, y, txt, edge in label_pts:
    if not txt:
        continue
    ax_tau.text(
        xhi + 0.02 * (tau_hi - tau_lo), y, txt,
        ha="left", va="center", color="black", fontsize=3.0, zorder=4,
    )
ax_tau.set_xlabel("Conditional effect (τ*)")
ax_tau.tick_params(axis="y", length=0)
ax_tau.spines[["top", "right", "left"]].set_visible(False)
ax_tau.legend(
    handles=tau_qtl_leg, loc="lower center", bbox_to_anchor=(0.5, 1.0),
    frameon=False, borderaxespad=0.0, ncol=3, handletextpad=0.5, columnspacing=0.9,
    fontsize=3.5,
)

# Panel d — stacked: colored lncRNA-restricted + white other CoS
# novel colored = novel-only; existing colored = existing-only ∪ novel+existing
edge_lw = 0.7
n_bio = len(gene_cos_rows_i)
gene_bar_w = 0.55
gene_counts = []
slice_fs = max(star_fs, 4.5)

for i, (key, _lab, _ids) in enumerate(gene_cos_rows_i):
    s = gene_cos_counts_i[key]
    edge = gene_type_palette[key]
    n_only, n_other = s["only"], s["other"]
    n_all = s["overall"]
    n_stack = n_only + n_other
    gene_counts.extend([n_only, n_other, n_stack, n_all])
    fill_only = (
        cos_slice_colors["novel_only"] if key == "novel_lncRNA"
        else cos_slice_colors["existing_only"]
    )
    ax_genes.bar(
        i, n_only, width=gene_bar_w,
        color=fill_only, edgecolor=edge, linewidth=edge_lw, zorder=3,
    )
    ax_genes.bar(
        i, n_other, bottom=n_only, width=gene_bar_w,
        color=cos_slice_colors["shared"], edgecolor=edge,
        linewidth=edge_lw, zorder=3,
    )
    # lncRNA-specific (colored slice) count at top of that slice
    if n_only > 0:
        ax_genes.text(
            i, n_only, f"{n_only:,}",
            va="bottom", ha="center", fontsize=slice_fs, color="black", zorder=5,
        )
    # overall at top of bar
    if n_all > 0:
        ax_genes.text(
            i, max(n_stack, n_all), f"{n_all:,}",
            va="bottom", ha="center", fontsize=slice_fs, color="black", zorder=5,
        )

pos = np.asarray([c for c in gene_counts if c and c > 0], dtype=float)
gene_ymin = max(0.8, float(pos.min()) * 0.55) if len(pos) else 0.8
gene_ymax = float(pos.max()) if len(pos) else 1.0
ax_genes.set_yscale("log", base=10)
ax_genes.set_xticks(range(n_bio))
ax_genes.set_xticklabels([lab for _, lab, _ in gene_cos_rows_i])
ax_genes.set_xlabel("")
ax_genes.set_ylabel("Unique genes w/ GWAS CoS", fontsize=5)
ax_genes.set_ylim(gene_ymin, gene_ymax * 2.4)
ax_genes.set_xlim(-0.55, n_bio - 0.45)
ax_genes.tick_params(axis="x", length=0, pad=2)
ax_genes.spines["top"].set_visible(False)
ax_genes.spines["right"].set_visible(False)
ax_genes.legend(
    handles=[
        Patch(facecolor=cos_slice_colors["novel_only"], edgecolor="black",
              linewidth=edge_lw, label="novel-only"),
        Patch(facecolor=cos_slice_colors["existing_only"], edgecolor="black",
              linewidth=edge_lw, label="lncRNA-only"),
        Patch(facecolor=cos_slice_colors["shared"], edgecolor="black",
              linewidth=edge_lw, label="other CoS"),
    ],
    loc="lower center", bbox_to_anchor=(0.5, 1.02),
    frameon=False, ncol=3, handletextpad=0.4, columnspacing=0.7, borderaxespad=0.0,
    fontsize=3.2,
)

color_all, color_gws = "#D0D0D0", "#6E6E6E"
scope_specs = [
    ("n_all", all_loci_i, color_all, 2, "p<1e-5"),
    ("n_gws", gws_loci_i, color_gws, 3, "GWS"),
]

combos = [tuple(c) for c in upset_combo_counts_i["combo"].tolist()]
x = np.arange(len(combos))

bar_w = 0.55
counts_all = upset_combo_counts_i["n_all"].to_numpy(dtype=float)
counts_gws = upset_combo_counts_i["n_gws"].to_numpy(dtype=float)
for col, _loci, color, z, _name in scope_specs:
    counts = upset_combo_counts_i[col].to_numpy(dtype=float)
    ax_e_bar.bar(
        x, counts, width=bar_w,
        color=color, edgecolor="black", linewidth=0.35, zorder=z,
    )
# 1e-5 on top; 5.7e-9 in white just inside the dark bar tip
for xi, n_all, n_gws in zip(x, counts_all, counts_gws):
    if n_all > 0:
        ax_e_bar.text(
            xi, n_all, f"{int(n_all):,}",
            ha="center", va="bottom", color="0.25", zorder=4, fontsize=2.6,
        )
    if n_gws > 0:
        ax_e_bar.text(
            xi, n_gws * 0.82, f"{int(n_gws):,}",
            ha="center", va="top", color="white", zorder=4, fontsize=2.6,
        )

pos = np.concatenate([counts_all, counts_gws])
pos = pos[pos > 0]
ymin = max(0.8, float(pos.min()) * 0.55) if len(pos) else 0.8
ymax = float(pos.max()) if len(pos) else 1.0
ax_e_bar.set_yscale("log", base=10)
ax_e_bar.set_ylim(ymin, ymax * 2.4 if ymax > 0 else 1)
ax_e_bar.set_ylabel("Cross-tissue\nGWAS-QTL CoS")
ax_e_bar.set_xticks([])
ax_e_bar.spines["top"].set_visible(False)
ax_e_bar.spines["right"].set_visible(False)
ax_e_bar.spines["bottom"].set_visible(False)
ax_e_bar.set_xlim(-0.7, len(x) - 0.3)
ax_e_leg.legend(
    handles=[
        Patch(facecolor=color_gws, edgecolor="black", linewidth=0.4,
              label=r"$p < 5.7 \times 10^{-9}$"),
        Patch(facecolor=color_all, edgecolor="black", linewidth=0.4,
              label=r"$p < 1 \times 10^{-5}$"),
    ],
    title=r"GWAS $p$-value",
    loc="center", frameon=False, ncol=1,
    handletextpad=0.5, labelspacing=0.4,
)
upset_row_order = list(upset_biotypes_i)
n_sets = len(upset_row_order)
y_pos = -np.arange(n_sets, dtype=float)
for j, combo in enumerate(combos):
    present_y = []
    for i, bt in enumerate(upset_row_order):
        on = bt in combo
        ax_e_dot.scatter(
            j, y_pos[i],
            s=28 if on else 14,
            color=gene_type_palette[bt] if on else "0.85",
            edgecolors="k" if on else "none",
            linewidths=0.3 if on else 0,
            zorder=3,
        )
        if on:
            present_y.append(y_pos[i])
    if len(present_y) >= 2:
        ax_e_dot.plot([j, j], [min(present_y), max(present_y)], color="0.25", lw=0.8, zorder=2)
    # Highlight boxes: novel-only = novel color; existing-only & novel+existing = existing color
    if combo == ("novel_lncRNA",):
        _box_edge = cos_slice_colors["novel_only"]
    elif combo in {("existing_lncRNA",), ("existing_lncRNA", "novel_lncRNA")}:
        _box_edge = cos_slice_colors["existing_only"]
    else:
        _box_edge = None
    if _box_edge is not None:
        pad_x, pad_y = 0.38, 0.48
        y0, y1 = y_pos.min() - pad_y, y_pos.max() + pad_y
        ax_e_dot.add_patch(Rectangle(
            (j - pad_x, y0), 2 * pad_x, y1 - y0,
            fill=False, edgecolor=_box_edge, linewidth=1.15, zorder=4,
        ))

ax_e_dot.set_xlim(-0.7, len(x) - 0.3)
ax_e_dot.set_ylim(y_pos.min() - 0.85, 0.85)
ax_e_dot.set_xticks([])
ax_e_dot.set_yticks(y_pos)
ax_e_dot.set_yticklabels([])
ax_e_dot.tick_params(axis="y", length=0, labelleft=False)
for sp in ax_e_dot.spines.values():
    sp.set_visible(False)

sizes_all = np.array([upset_set_sizes_i[bt] for bt in upset_row_order], dtype=float)
sizes_gws = np.array([upset_set_sizes_gws_i[bt] for bt in upset_row_order], dtype=float)
bar_h = 0.55
pos = np.concatenate([sizes_all, sizes_gws])
pos = pos[pos > 0]
x_min = max(10.0, float(pos.min()) * 0.35) if len(pos) else 10.0
x_max = float(pos.max()) if len(pos) else 1e4
for y, n_all, n_gws in zip(y_pos, sizes_all, sizes_gws):
    ax_e_set.barh(
        y, n_all, height=bar_h,
        color=color_all, edgecolor="k", linewidth=0.35, zorder=2,
    )
    ax_e_set.barh(
        y, n_gws, height=bar_h,
        color=color_gws, edgecolor="k", linewidth=0.35, zorder=3,
    )
    if n_all > 0:
        ax_e_set.text(
            n_all, y, f"{int(n_all):,} ",
            ha="right", va="center", color="0.25", zorder=4, fontsize=2.8,
        )
    if n_gws > 0:
        ax_e_set.text(
            n_gws, y, f" {int(n_gws):,}",
            ha="left", va="center", color="white", zorder=4, fontsize=2.8,
        )
ax_e_set.set_ylim(ax_e_dot.get_ylim())
ax_e_set.set_xscale("log", base=10)
ax_e_set.set_xlim(x_max * 2.2, x_min)  # inverted: bars grow left toward larger counts
ax_e_set.set_xticks([1e2, 1e3, 1e4])
ax_e_set.set_xticklabels([r"$10^{2}$", r"$10^{3}$", r"$10^{4}$"])
ax_e_set.tick_params(axis="x", length=2, pad=1)
ax_e_set.set_yticks(y_pos)
# Labels between set-size bars and the intersection dots
ax_e_set.set_yticklabels([upset_set_labels_i[bt] for bt in upset_row_order], fontsize=4.5)
ax_e_set.yaxis.tick_right()
ax_e_set.tick_params(axis="y", length=0, pad=3, labelright=True, labelleft=False)
ax_e_set.set_xlabel("Cross-tissue\nGWAS-QTL CoS", labelpad=1)
for sp in ("top", "right", "left"):
    ax_e_set.spines[sp].set_visible(False)

plot_df = or_df_i
cats = list(cat_order_i)
groups = list(or_groups_i)
y_base = {lab: i for i, lab in enumerate(cats)}
n_grp = len(groups)
if n_grp <= 1:
    offsets = {g: 0.0 for g in groups}
else:
    offsets = {
        g: -0.22 + i * (0.44 / (n_grp - 1))
        for i, g in enumerate(groups)
    }
or_lo_min, or_hi_max = 1.0, 1.0
plot_idx = plot_df.set_index(["group", "label"])
for group in groups:
    color = modality_or_colors[group]
    marker = modality_or_markers[group]
    mfc = modality_or_face.get(group, color)
    mec = modality_or_edge.get(group, "k")
    mew = 0.6 if group == share_shared else 0.3
    xs, ys, xlo, xhi, labs = [], [], [], [], []
    for lab in cats:
        y = y_base[lab] + offsets[group]
        if (group, lab) not in plot_idx.index:
            ax_or.plot(1.0, y, marker="x", color="0.55", ms=3.0, mew=0.8, zorder=3)
            continue
        row = plot_idx.loc[(group, lab)]
        if isinstance(row, pd.DataFrame):
            row = row.iloc[0]
        or_hat, lo, hi = row["OR"], row["CI_low"], row["CI_high"]
        if not (np.isfinite(or_hat) and np.isfinite(lo) and np.isfinite(hi)):
            ax_or.plot(1.0, y, marker="x", color="0.55", ms=3.0, mew=0.8, zorder=3)
            continue
        xs.append(or_hat)
        ys.append(y)
        xlo.append(lo)
        xhi.append(hi)
        labs.append(sig_stars(row["pvalue"]))
        or_lo_min = min(or_lo_min, lo)
        or_hi_max = max(or_hi_max, hi)
    if not xs:
        continue
    ax_or.errorbar(
        xs, ys,
        xerr=[np.array(xs) - np.array(xlo), np.array(xhi) - np.array(xs)],
        fmt=marker, color=color, ecolor=color,
        markerfacecolor=mfc, markeredgecolor=mec, markeredgewidth=mew,
        elinewidth=0.75, markersize=2.4, label=group, zorder=3,
    )
    for x, y, s, hi in zip(xs, ys, labs, xhi):
        if s:
            ax_or.text(hi * 1.08, y, s, ha="left", va="center", color="black", fontsize=star_fs)
ax_or.axvline(1, color="0.5", ls="-", lw=0.5, zorder=1)
for _y in range(1, len(cats)):
    ax_or.axhline(_y - 0.5, color="0.75", ls="-", lw=0.25, zorder=0)
ax_or.set_yticks(list(y_base.values()))
ax_or.set_yticklabels(
    list(y_base.keys()), ha="right", va="center",
    linespacing=0.85, fontsize=4.5,
)
ax_or.tick_params(axis="y", pad=1, length=0)
ax_or.set_xlabel(or_xlabel_i)
ax_or.set_xscale("log", base=2)
lo = max(or_lo_min, 1e-6)
hi = max(or_hi_max, 1.0)
pad = 1.25
x0, x1 = lo / pad, hi * pad * 1.15
k0 = int(np.floor(np.log2(x0)))
k1 = int(np.ceil(np.log2(x1)))
ticks = [2**k for k in range(k0, k1 + 1)]
ax_or.set_xticks(ticks)
ax_or.set_xticklabels([rf"$2^{{{k}}}$" if k != 0 else "1" for k in range(k0, k1 + 1)])
ax_or.set_xlim(x0, x1)
ax_or.set_ylim(len(cats) - 0.4, -0.6)  # first category at top
ax_or.spines["top"].set_visible(False)
ax_or.spines["right"].set_visible(False)
ax_or.spines["left"].set_visible(False)
ax_or.legend(
    loc="lower center", bbox_to_anchor=(0.5, 1.02), frameon=False,
    ncol=len(groups), handletextpad=0.4, columnspacing=1.0, fontsize=3.5,
)

# Panel letters: a/b/d share y; a/c share x; d/e share x
_inv = fig.transFigure.inverted()

def _ax_fig(ax, x, y):
    return _inv.transform(ax.transAxes.transform((x, y)))

_xa, _ya = _ax_fig(ax_enrich, -0.18, 1.02)
_xb, _ = _ax_fig(ax_tau, -0.08, 1.02)
_xd, _yd = _ax_fig(ax_genes, -0.28, 1.02)
_, _ye = _ax_fig(ax_or, -0.28, 1.02)
_, _yc = _ax_fig(ax_e_bar, 0, 1.02)
_y_abd = max(_ya, _yd)  # a/b/d horizontally aligned
_x_ac = _xa                 # a/c vertically aligned
_x_de = _xd                 # d/e vertically aligned
for _x, _y, _lab in [
    (_x_ac, _y_abd, "a"), (_xb, _y_abd, "b"), (_x_ac, _yc, "c"),
    (_x_de, _y_abd, "d"), (_x_de, _ye, "e"),
]:
    fig.text(_x, _y, _lab, fontweight="bold", va="bottom", ha="right", fontsize=9)

plt.show()


## Save figure


In [ ]:
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_pdf, bbox_inches="tight")
fig.savefig(fig_png, bbox_inches="tight")
print(f"Wrote {fig_pdf}")
print(f"Wrote {fig_png}")


## Print stats


In [ ]:
print(f"GWAS traits: {n_traits}")

# Heritability enrichment (marginal models, testis excluded; novel lncRNA sQTL not tested)
for biotype, q, label in [
    ("protein_coding", "eqtl", "protein-coding eVariants"),
    ("lncRNA_all", "eqtl", "all lncRNA eVariants"),
    ("novel_lncRNA", "eqtl", "novel lncRNA eVariants"),
    ("existing_lncRNA", "eqtl", "existing lncRNA eVariants"),
    ("protein_coding", "sqtl", "protein-coding sVariants"),
]:
    m = meta(model_id(biotype, "marginal", q))
    print(f"{label} enrichment: {m['mu']:.1f}-fold, p = {m['p_vs1']:.1e}")
print("novel lncRNA sVariants: excluded from S-LDSC")

# Conditional effect (tau*)
for biotype, scope, q, label in [
    ("lncRNA_all", "base_pc_esqtl", "eqtl", "lncRNA eVariants (alongside protein-coding e/sVariants)"),
    ("lncRNA_all", "base_pc_esqtl", "sqtl", "lncRNA sVariants (alongside protein-coding e/sVariants)"),
    ("protein_coding", "marginal", "eqtl", "protein-coding eVariants (with baseline annotations)"),
    ("protein_coding", "marginal", "sqtl", "protein-coding sVariants (with baseline annotations)"),
    ("protein_coding_downsampled", "marginal", "eqtl", "downsampled protein-coding eVariants (with baseline annotations)"),
]:
    m = meta_tau_star(model_id(biotype, scope, q))
    print(f"{label} tau*: {m['mu']:.2f}, p = {m['p_gt0']:.1e}")

# Genes with GWAS-QTL colocalization (p < 1e-5)
print(f"novel lncRNA genes with colocalization: {gene_cos_counts['novel_lncRNA']['overall']:,}")
print(f"novel lncRNA-specific colocalization genes: {gene_cos_counts['novel_lncRNA']['only']:,}")
print(f"existing lncRNA genes with colocalization: {gene_cos_counts['existing_lncRNA']['overall']:,}")

# Panel e odds ratios (ascertainment-adjusted) cited in Results
for trait, label in [("tc_nontestis", "tissue-concordant expression"), ("dev_expr", "developmental expression")]:
    for group in panel_e_group_order:
        r = sharing_or_df[
            (sharing_or_df["trait"] == trait) & (sharing_or_df["group"] == group)
        ].iloc[0]
        print(f"{label}, OR {group} colocalization = {r['OR']:.2f}, p = {r['pvalue']:.1e}")
